#Modelling: Baseline and Class Weights

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

X_train = pd.read_csv("/content/drive/MyDrive/X_train.csv")
X_test = pd.read_csv("/content/drive/MyDrive/X_test.csv")
y_train = pd.read_csv("/content/drive/MyDrive/y_train.csv")["Class"]
y_test = pd.read_csv("/content/drive/MyDrive/y_test.csv")["Class"]

print(X_train.shape, X_test.shape)

(226980, 30) (56746, 30)


An Evaluation Function (reused for every model)

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

def evaluate(model, name):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(name)
    print(classification_report(y_test, pred, digits=3))
    print(confusion_matrix(y_test, pred))
    print("PR-AUC:", round(average_precision_score(y_test, proba), 3))

Baseline model: Logistic Regression with no imbalance handling.

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
evaluate(lr, "Logistic Regression (baseline)")

Logistic Regression (baseline)
              precision    recall  f1-score   support

           0      0.999     1.000     1.000     56651
           1      0.848     0.589     0.696        95

    accuracy                          0.999     56746
   macro avg      0.924     0.795     0.848     56746
weighted avg      0.999     0.999     0.999     56746

[[56641    10]
 [   39    56]]
PR-AUC: 0.692


Same model with class_weight="balanced", so errors on fraud cost more

In [13]:
lr_w = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_w.fit(X_train, y_train)
evaluate(lr_w, "Logistic Regression (class weights)")

Logistic Regression (class weights)
              precision    recall  f1-score   support

           0      1.000     0.975     0.987     56651
           1      0.056     0.874     0.106        95

    accuracy                          0.975     56746
   macro avg      0.528     0.925     0.547     56746
weighted avg      0.998     0.975     0.986     56746

[[55258  1393]
 [   12    83]]
PR-AUC: 0.672


## Summary

- *Baseline (no imbalance handling):* precision 0.848, recall 0.589, F1 0.696, PR-AUC 0.692. It rarely raises false alarms but misses 39 of 95 frauds.
- *Class weights = "balanced":* recall rises to 0.874 (12 missed frauds), but precision drops to 0.056 with 1,393 false alarms. PR-AUC is similar (0.672).
- *Takeaway:* class weights trade precision for recall without improving overall ranking quality. Accuracy (~99.9%) is misleading on this data, so recall, precision and PR-AUC are used instead.

*Next step:* try SMOTE (on the training set only) and stronger models (Random Forest, XGBoost).